# LeetCode 392 — Is Subsequence bằng Python

**Link đề:** https://leetcode.com/problems/is-subsequence/
**Độ khó:** Easy · **Chủ đề:** Two Pointers, String, Dynamic Programming

```python
class Solution:
    def isSubsequence(self, s: str, t: str) -> bool:
        ...
```

## Mục tiêu học
1. Phân biệt **dãy con (subsequence)** với **chuỗi con liên tiếp (substring)**.
2. Nắm kỹ thuật **hai con trỏ (two pointers)** + **tham lam (greedy)**: khớp ký tự **sớm nhất có thể**.
3. Biết thêm: `str.find` với vị trí bắt đầu, mẹo **iterator** của Python, và lời giải **quy hoạch động (DP)**.
4. Giải **Follow up** (rất nhiều chuỗi `s` cần kiểm tra trên cùng `t`): tiền xử lý **danh sách vị trí** + **tìm nhị phân**.
5. Trace hai con trỏ từng bước; chứng minh vì sao tham lam không bỏ sót.

## 1. Đề bài

Cho hai chuỗi `s` và `t`. Trả `True` nếu `s` là **dãy con** của `t`.

**Dãy con** của một chuỗi được tạo bằng cách **xóa một số ký tự (có thể không xóa gì)** mà **không đổi thứ tự** các ký tự còn lại. Ví dụ `"ace"` là dãy con của `"abcde"`, còn `"aec"` thì không (sai thứ tự).

| `s` | `t` | Kết quả | Vì sao |
|---|---|:---:|---|
| `"abc"` | `"ahbgdc"` | `True` | a̲ h b̲ g d c̲ |
| `"axc"` | `"ahbgdc"` | `False` | không có `x` |
| `""` | `"abc"` | `True` | chuỗi rỗng là dãy con của mọi chuỗi |
| `"ace"` | `"abcde"` | `True` | không cần liên tiếp |
| `"aec"` | `"abcde"` | `False` | `e` đứng sau `c` trong `t` |

**Constraints:** `0 <= len(s) <= 100`; `0 <= len(t) <= 10^4`; chỉ chữ thường.

**Follow up:** có rất nhiều chuỗi `s1, s2, ..., sk` với `k >= 10^9` cần kiểm tra lần lượt trên cùng `t`. Bạn sẽ thay đổi code thế nào?

In [1]:
# Substring (liên tiếp) khác subsequence (giữ thứ tự, có thể nhảy cóc)
t = "abcde"
print("'ace' in t (substring)?", "ace" in t)
print("'bcd' in t (substring)?", "bcd" in t)
print("Nhưng 'ace' là subsequence: lấy a(0), c(2), e(4)")

'ace' in t (substring)? False
'bcd' in t (substring)? True
Nhưng 'ace' là subsequence: lấy a(0), c(2), e(4)


## 2. Nền tảng Python

### 2.1. Chỉ số và vòng `while` với hai biến

Hai con trỏ là hai **chỉ số** `i` (trong `s`) và `j` (trong `t`) di chuyển độc lập. `while i < len(s) and j < len(t):` chạy khi **cả hai** còn trong biên.

In [2]:
s, t = "ab", "xaxb"
i = j = 0
while i < len(s) and j < len(t):
    print(f"i={i} (s[i]={s[i]!r}), j={j} (t[j]={t[j]!r})")
    if s[i] == t[j]:
        i += 1
    j += 1
print("Kết thúc: i =", i, "| len(s) =", len(s))

i=0 (s[i]='a'), j=0 (t[j]='x')
i=0 (s[i]='a'), j=1 (t[j]='a')
i=1 (s[i]='b'), j=2 (t[j]='x')
i=1 (s[i]='b'), j=3 (t[j]='b')
Kết thúc: i = 2 | len(s) = 2


### 2.2. `str.find(sub, start)` — tìm từ vị trí `start`

`t.find(ch, start)` trả chỉ số **đầu tiên ≥ `start`** chứa `ch`, hoặc `-1`.

In [3]:
t = "ahbgdc"
print(t.find("b"), t.find("b", 3), t.find("c", 2), t.find("x"))

2 -1 5 -1


### 2.3. Iterator — "con trỏ ẩn" của Python

`it = iter(t)` tạo một **bộ duyệt** nhớ vị trí hiện tại. `ch in it` **tiêu thụ** các phần tử cho tới khi gặp `ch` (hoặc hết). Lần `in` tiếp theo **tiếp tục từ chỗ đó**, không quay lại đầu. Đây chính là hai con trỏ được "giấu" trong iterator.

In [4]:
it = iter("ahbgdc")
print("'b' in it:", "b" in it)    # tiêu thụ a, h, b
print("'a' in it:", "a" in it)    # a đã qua rồi -> tiêu thụ hết g, d, c -> False
print("phần còn lại:", list(it))  # rỗng

'b' in it: True
'a' in it: False
phần còn lại: []


## 3. Cách 1 — Trực giác (vét cạn): sinh mọi dãy con của `t`?

Ý nghĩ đầu tiên: liệt kê mọi dãy con của `t` rồi xem `s` có nằm trong đó không. Nhưng `t` dài `m` có tới `2^m` dãy con (mỗi ký tự chọn **lấy** hoặc **bỏ**) → với `m = 10^4` là không tưởng. Chỉ dùng được để **kiểm thử** với chuỗi rất ngắn.

`itertools.combinations(range(m), k)` sinh mọi bộ `k` chỉ số **tăng dần** — đúng định nghĩa giữ thứ tự.

In [5]:
from itertools import combinations

def is_subseq_bruteforce(s: str, t: str) -> bool:
    k = len(s)
    for idxs in combinations(range(len(t)), k):
        if all(t[p] == s[q] for q, p in enumerate(idxs)):
            return True
    return k == 0

print(list(combinations(range(4), 2)))
for s, t in [("abc", "ahbgdc"), ("axc", "ahbgdc"), ("", "abc"), ("aec", "abcde")]:
    print(repr(s), repr(t), "->", is_subseq_bruteforce(s, t))
print("Số dãy con của chuỗi dài 20:", 2 ** 20, "| dài 10^4: 2^10000 (quá lớn)")

[(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)]
'abc' 'ahbgdc' -> True
'axc' 'ahbgdc' -> False
'' 'abc' -> True
'aec' 'abcde' -> False
Số dãy con của chuỗi dài 20: 1048576 | dài 10^4: 2^10000 (quá lớn)


### 3.1. Nút thắt và insight tham lam

Vét cạn thử **mọi cách chọn vị trí**. Nhưng ta chỉ cần **một** cách. Quan sát: khi khớp `s[0]`, nên chọn lần xuất hiện **sớm nhất** trong `t` — vì chọn sớm thì phần còn lại của `t` **dài nhất**, cơ hội khớp các ký tự sau **không thể tệ hơn**.

⇒ Duyệt `t` một lần từ trái sang phải; gặp ký tự bằng `s[i]` thì "ăn" ngay và chuyển sang `s[i+1]`. Đây là **tham lam + hai con trỏ**.

## 4. Cách 2 — Hai con trỏ (lời giải chính)

**Đoạn 1 — khởi tạo:** `i = 0` (ký tự cần khớp tiếp theo của `s`), `j = 0` (vị trí đang đọc trong `t`).

**Đoạn 2 — vòng lặp:** khi `i < len(s)` và `j < len(t)`:
- nếu `s[i] == t[j]` → khớp được → `i += 1`;
- luôn `j += 1` (đọc ký tự tiếp theo của `t`).

**Đoạn 3 — kết luận:** `s` là dãy con ⇔ đã khớp hết, tức `i == len(s)`.

In [6]:
def is_subseq_two_pointers(s: str, t: str) -> bool:
    i = j = 0
    while i < len(s) and j < len(t):
        if s[i] == t[j]:
            i += 1
        j += 1
    return i == len(s)

for s, t in [("abc", "ahbgdc"), ("axc", "ahbgdc"), ("", "abc"), ("ace", "abcde"), ("aec", "abcde"), ("a", "")]:
    print(repr(s), repr(t), "->", is_subseq_two_pointers(s, t))

'abc' 'ahbgdc' -> True
'axc' 'ahbgdc' -> False
'' 'abc' -> True
'ace' 'abcde' -> True
'aec' 'abcde' -> False
'a' '' -> False


## 5. TRACE — `s = "abc"`, `t = "ahbgdc"`

| Bước | `i` | `s[i]` | `j` | `t[j]` | Khớp? | Sau bước |
|---:|---:|:-:|---:|:-:|:-:|---|
| 1 | 0 | a | 0 | a | ✔ | i=1, j=1 |
| 2 | 1 | b | 1 | h | ✘ | i=1, j=2 |
| 3 | 1 | b | 2 | b | ✔ | i=2, j=3 |
| 4 | 2 | c | 3 | g | ✘ | i=2, j=4 |
| 5 | 2 | c | 4 | d | ✘ | i=2, j=5 |
| 6 | 2 | c | 5 | c | ✔ | i=3, j=6 |

Dừng vì `i = 3 = len(s)` → `True`. Với `s = "axc"`: `x` không bao giờ khớp, `j` chạy hết `t` trong khi `i` kẹt ở 1 → `False`.

In [7]:
def trace_subseq(s: str, t: str) -> bool:
    i = j = 0
    step = 1
    print(f"s={s!r}, t={t!r}")
    while i < len(s) and j < len(t):
        match = s[i] == t[j]
        print(f"  Bước {step}: i={i} s[i]={s[i]!r} | j={j} t[j]={t[j]!r} -> {'KHỚP, i += 1' if match else 'bỏ qua t[j]'}")
        if match:
            i += 1
        j += 1
        step += 1
    reason = "đã khớp hết s" if i == len(s) else f"hết t nhưng còn {len(s) - i} ký tự của s chưa khớp"
    print(f"  Dừng: i={i}, j={j} -> {reason} -> {i == len(s)}")
    return i == len(s)

_ = trace_subseq("abc", "ahbgdc")
_ = trace_subseq("axc", "ahbgdc")

s='abc', t='ahbgdc'
  Bước 1: i=0 s[i]='a' | j=0 t[j]='a' -> KHỚP, i += 1
  Bước 2: i=1 s[i]='b' | j=1 t[j]='h' -> bỏ qua t[j]
  Bước 3: i=1 s[i]='b' | j=2 t[j]='b' -> KHỚP, i += 1
  Bước 4: i=2 s[i]='c' | j=3 t[j]='g' -> bỏ qua t[j]
  Bước 5: i=2 s[i]='c' | j=4 t[j]='d' -> bỏ qua t[j]
  Bước 6: i=2 s[i]='c' | j=5 t[j]='c' -> KHỚP, i += 1
  Dừng: i=3, j=6 -> đã khớp hết s -> True
s='axc', t='ahbgdc'
  Bước 1: i=0 s[i]='a' | j=0 t[j]='a' -> KHỚP, i += 1
  Bước 2: i=1 s[i]='x' | j=1 t[j]='h' -> bỏ qua t[j]
  Bước 3: i=1 s[i]='x' | j=2 t[j]='b' -> bỏ qua t[j]
  Bước 4: i=1 s[i]='x' | j=3 t[j]='g' -> bỏ qua t[j]
  Bước 5: i=1 s[i]='x' | j=4 t[j]='d' -> bỏ qua t[j]
  Bước 6: i=1 s[i]='x' | j=5 t[j]='c' -> bỏ qua t[j]
  Dừng: i=1, j=6 -> hết t nhưng còn 2 ký tự của s chưa khớp -> False


## 6. Cách 3 — `str.find` nhảy thẳng tới lần xuất hiện kế tiếp

Thay vì tăng `j` từng bước, dùng `t.find(ch, pos)` để **nhảy** tới vị trí sớm nhất ≥ `pos` chứa `ch`. Cùng ý tưởng tham lam, nhưng `find` chạy bằng C nên nhanh hơn trên thực tế.

In [8]:
def is_subseq_find(s: str, t: str) -> bool:
    pos = 0
    for ch in s:
        k = t.find(ch, pos)
        if k == -1:
            return False
        pos = k + 1            # ký tự tiếp theo phải đứng SAU k
    return True

for s, t in [("abc", "ahbgdc"), ("axc", "ahbgdc"), ("", "abc"), ("aa", "ab")]:
    print(repr(s), repr(t), "->", is_subseq_find(s, t))

'abc' 'ahbgdc' -> True
'axc' 'ahbgdc' -> False
'' 'abc' -> True
'aa' 'ab' -> False


## 7. Cách 4 — Mẹo iterator một dòng

```python
it = iter(t)
return all(ch in it for ch in s)
```

Mỗi `ch in it` tiêu thụ `t` tới khi gặp `ch` — chính là con trỏ `j`. `all` dừng ngay khi có một `ch` không tìm được.

In [9]:
def is_subseq_iter(s: str, t: str) -> bool:
    it = iter(t)
    return all(ch in it for ch in s)

for s, t in [("abc", "ahbgdc"), ("axc", "ahbgdc"), ("", "abc"), ("aec", "abcde")]:
    print(repr(s), repr(t), "->", is_subseq_iter(s, t))

'abc' 'ahbgdc' -> True
'axc' 'ahbgdc' -> False
'' 'abc' -> True
'aec' 'abcde' -> False


## 8. Cách 5 — Quy hoạch động (DP)

Đặt `dp[i][j]` = `True` nếu `s[:i]` (i ký tự đầu của `s`) là dãy con của `t[:j]`.

- `dp[0][j] = True` (chuỗi rỗng là dãy con của mọi tiền tố).
- `dp[i][0] = False` với `i > 0`.
- Chuyển trạng thái: nếu `s[i-1] == t[j-1]` thì `dp[i][j] = dp[i-1][j-1]` (khớp hai ký tự cuối) **hoặc** `dp[i][j-1]` (không dùng `t[j-1]`); ngược lại `dp[i][j] = dp[i][j-1]`.

Đáp án `dp[len(s)][len(t)]`. Chi phí `O(n·m)` — chậm hơn hai con trỏ, nhưng là nền tảng cho bài khó hơn (**115 Distinct Subsequences** — đếm số cách, **1143 Longest Common Subsequence**).

In [10]:
def is_subseq_dp(s: str, t: str) -> bool:
    n, m = len(s), len(t)
    dp = [[False] * (m + 1) for _ in range(n + 1)]   # bảng (n+1) x (m+1)
    for j in range(m + 1):
        dp[0][j] = True
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            if s[i - 1] == t[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] or dp[i][j - 1]
            else:
                dp[i][j] = dp[i][j - 1]
    return dp[n][m]

def show_dp(s, t):
    n, m = len(s), len(t)
    dp = [[False] * (m + 1) for _ in range(n + 1)]
    for j in range(m + 1):
        dp[0][j] = True
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            dp[i][j] = (dp[i - 1][j - 1] or dp[i][j - 1]) if s[i - 1] == t[j - 1] else dp[i][j - 1]
    print("      ''  " + "  ".join(t))
    for i in range(n + 1):
        label = "''" if i == 0 else s[i - 1] + " "
        print(f"  {label}  " + "  ".join("T" if v else "." for v in dp[i]))
    return dp[n][m]

print(show_dp("abc", "ahbgdc"))

      ''  a  h  b  g  d  c
  ''  T  T  T  T  T  T  T
  a   .  T  T  T  T  T  T
  b   .  .  .  T  T  T  T
  c   .  .  .  .  .  .  T
True


**Chú ý:** `[[False] * (m + 1) for _ in range(n + 1)]` tạo `n + 1` **list riêng biệt**. Viết `[[False] * (m + 1)] * (n + 1)` sẽ tạo `n + 1` **tham chiếu tới cùng một list** — sửa một hàng là sửa tất cả (lỗi kinh điển).

Đọc bảng: hàng `c` cột `c` cuối cùng là `T` → `True`. Mỗi hàng `i` "bật" `T` từ cột đầu tiên mà `s[:i]` khớp được, và giữ `T` về bên phải.

## 9. Follow up — Rất nhiều `s` trên cùng một `t`

Nếu có `k ≥ 10^9` chuỗi `s`, mỗi lần quét lại `t` (`O(m)`) là quá đắt. Ta **tiền xử lý `t` một lần**:

`positions[c]` = danh sách **tăng dần** các chỉ số mà ký tự `c` xuất hiện trong `t`.

Khi kiểm tra một `s`: giữ `pos` = vị trí tối thiểu được phép. Với mỗi `ch`, tìm nhị phân trong `positions[ch]` phần tử **đầu tiên ≥ `pos`** (`bisect_left`). Không có → `False`; có ở `k` → `pos = k + 1`.

Mỗi `s` chỉ tốn `O(n log m)` thay vì `O(m)`.

In [11]:
from bisect import bisect_left
from collections import defaultdict

class SubsequenceChecker:
    def __init__(self, t: str):
        self.positions = defaultdict(list)
        for idx, ch in enumerate(t):
            self.positions[ch].append(idx)       # tự động tăng dần

    def check(self, s: str) -> bool:
        pos = 0
        for ch in s:
            lst = self.positions.get(ch)
            if not lst:
                return False
            k = bisect_left(lst, pos)            # chỉ số trong lst của vị trí đầu tiên >= pos
            if k == len(lst):
                return False
            pos = lst[k] + 1
        return True

checker = SubsequenceChecker("ahbgdcab")
print(dict(checker.positions))
for s in ["abc", "axc", "", "aa", "bb", "ca", "cba"]:
    print(repr(s), "->", checker.check(s))

{'a': [0, 6], 'h': [1], 'b': [2, 7], 'g': [3], 'd': [4], 'c': [5]}
'abc' -> True
'axc' -> False
'' -> True
'aa' -> True
'bb' -> True
'ca' -> True
'cba' -> False


### 9.1. Biến thể của Follow up: bảng "vị trí kế tiếp" (`next_pos`)

Tiền xử lý bảng `nxt[j][c]` = vị trí đầu tiên `≥ j` của chữ `c` trong `t` (hoặc `m` nếu không có). Tốn `O(26·m)` bộ nhớ, nhưng mỗi `s` chỉ còn `O(n)` — **không cần tìm nhị phân**. Dựng bảng từ phải sang trái: `nxt[j] = nxt[j+1]` rồi sửa ô của `t[j]`.

In [12]:
def build_next(t: str):
    m = len(t)
    nxt = [None] * (m + 1)
    nxt[m] = [m] * 26                       # sau cuối: không còn chữ nào
    for j in range(m - 1, -1, -1):          # range lùi: m-1, m-2, ..., 0
        row = nxt[j + 1][:]                 # sao chép hàng kế tiếp
        row[ord(t[j]) - 97] = j
        nxt[j] = row
    return nxt

def check_with_next(nxt, s: str) -> bool:
    m = len(nxt) - 1
    j = 0
    for ch in s:
        if j > m:
            return False
        k = nxt[j][ord(ch) - 97]
        if k == m:
            return False
        j = k + 1
    return True

nxt = build_next("ahbgdc")
print("nxt[0] cho a,b,c:", nxt[0][:3], "| nxt[3] cho a,b,c:", nxt[3][:3])
print([check_with_next(nxt, s) for s in ["abc", "axc", "", "acb"]])

nxt[0] cho a,b,c: [0, 2, 5] | nxt[3] cho a,b,c: [6, 6, 5]
[True, False, True, False]


`range(m - 1, -1, -1)` đếm lùi từ `m - 1` về `0` (điểm dừng `-1` không tính). `row[:]` là bản sao nông của list.

## 10. Code nộp LeetCode

In [13]:
class Solution:
    def isSubsequence(self, s: str, t: str) -> bool:
        i = j = 0
        while i < len(s) and j < len(t):
            if s[i] == t[j]:
                i += 1
            j += 1
        return i == len(s)


sol = Solution()
print(sol.isSubsequence("abc", "ahbgdc"), sol.isSubsequence("axc", "ahbgdc"))

True False


Bản ngắn (iterator):

```python
class Solution:
    def isSubsequence(self, s: str, t: str) -> bool:
        it = iter(t)
        return all(ch in it for ch in s)
```

## 11. Vì sao đúng? + Độ phức tạp

**Tham lam đúng (lập luận "đổi chỗ"):** giả sử `s` là dãy con của `t` theo các vị trí `p0 < p1 < ... < p(n-1)`. Thuật toán chọn `g0` = vị trí **sớm nhất** có `t[g0] = s[0]`, nên `g0 ≤ p0`. Thay `p0` bằng `g0` vẫn hợp lệ (vì `g0 ≤ p0 < p1`). Lặp lại cho `s[1]` trên phần `t` sau `g0`: lựa chọn tham lam `g1 ≤ p1`, ... ⇒ thuật toán **luôn tìm được** cách khớp nếu tồn tại. Ngược lại, nếu thuật toán khớp hết thì rõ ràng đã chỉ ra một cách khớp → `True` đúng.

**Invariant:** *sau mỗi bước, `s[:i]` là dãy con của `t[:j]`, và `i` là số ký tự lớn nhất có thể khớp trong `t[:j]`.*

Gọi `n = len(s)`, `m = len(t)`, `k` = số chuỗi `s` (Follow up).

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn dãy con | `O(C(m, n)·n)` — hàm mũ | `O(n)` |
| 2. Hai con trỏ | `O(n + m)` → thực chất `O(m)` | `O(1)` |
| 3. `str.find` | `O(m)` (nhảy trong C) | `O(1)` |
| 4. Iterator | `O(m)` | `O(1)` |
| 5. DP | `O(n·m)` | `O(n·m)` |
| Follow up: vị trí + bisect | tiền xử lý `O(m)`; mỗi `s`: `O(n log m)` | `O(m)` |
| Follow up: bảng `next` | tiền xử lý `O(26·m)`; mỗi `s`: `O(n)` | `O(26·m)` |

## 12. Bộ test: mẫu đề, biên, phản ví dụ, đối chiếu

In [14]:
methods = {
    "bruteforce": is_subseq_bruteforce,
    "two_pointers": is_subseq_two_pointers,
    "find": is_subseq_find,
    "iter": is_subseq_iter,
    "dp": is_subseq_dp,
    "checker": lambda s, t: SubsequenceChecker(t).check(s),
    "next_table": lambda s, t: check_with_next(build_next(t), s),
    "Solution": Solution().isSubsequence,
}
cases = [
    ("abc", "ahbgdc", True), ("axc", "ahbgdc", False),     # mẫu đề
    ("", "", True), ("", "abc", True), ("a", "", False),   # rỗng
    ("ace", "abcde", True), ("aec", "abcde", False),       # thứ tự
    ("aa", "a", False), ("aa", "aba", True),               # lặp ký tự
    ("abc", "abc", True), ("abcd", "abc", False),          # s dài hơn t
    ("b", "abc", True), ("c", "ab", False),
]
for s, t, exp in cases:
    for name, f in methods.items():
        assert f(s, t) == exp, (name, s, t)
    print(f"{s!r:>7} in {t!r:<9} -> {exp} PASS")

  'abc' in 'ahbgdc'  -> True PASS
  'axc' in 'ahbgdc'  -> False PASS
     '' in ''        -> True PASS
     '' in 'abc'     -> True PASS
    'a' in ''        -> False PASS
  'ace' in 'abcde'   -> True PASS
  'aec' in 'abcde'   -> False PASS
   'aa' in 'a'       -> False PASS
   'aa' in 'aba'     -> True PASS
  'abc' in 'abc'     -> True PASS
 'abcd' in 'abc'     -> False PASS
    'b' in 'abc'     -> True PASS
    'c' in 'ab'      -> False PASS


In [15]:
import random
rng = random.Random(392)
for _ in range(1500):
    t = "".join(rng.choice("abc") for _ in range(rng.randint(0, 9)))
    s = "".join(rng.choice("abc") for _ in range(rng.randint(0, 4)))
    exp = is_subseq_bruteforce(s, t)
    for name, f in methods.items():
        assert f(s, t) == exp, (name, s, t)
big_t = "".join(rng.choice("abcdefghijklmnopqrstuvwxyz") for _ in range(10**4))
big_s = big_t[::97][:100]                    # lấy mỗi ký tự thứ 97 -> chắc chắn là dãy con
assert Solution().isSubsequence(big_s, big_t) and SubsequenceChecker(big_t).check(big_s)
print("1500 test ngẫu nhiên + test lớn PASS")

1500 test ngẫu nhiên + test lớn PASS


`big_t[::97]` là slicing với **bước 97**: lấy ký tự ở chỉ số 0, 97, 194, ... — luôn là dãy con của `big_t`.

## 13. Lỗi người mới hay gặp

### Lỗi 1 — Dùng `in` (substring) thay vì subsequence
`"ace" in "abcde"` là `False` nhưng đáp án là `True`.

### Lỗi 2 — Chỉ kiểm tra "mọi ký tự có mặt", bỏ qua thứ tự
`all(ch in t for ch in s)` cho `"aec"` / `"abcde"` → `True` sai. Cũng sai về số lần (`"aa"` / `"a"`).

### Lỗi 3 — Dùng `t.find(ch)` không có `start`
Luôn tìm từ đầu → bỏ qua ràng buộc thứ tự.

### Lỗi 4 — Kết luận bằng `j == len(t)` thay vì `i == len(s)`
`j` chạy hết `t` cả khi khớp lẫn không khớp; điều kiện đúng là **đã khớp hết `s`**.

In [16]:
def wrong_substring(s, t):
    return s in t

def wrong_presence(s, t):
    return all(ch in t for ch in s)

def wrong_find_no_start(s, t):
    last = -1
    for ch in s:
        k = t.find(ch)                 # luôn tìm từ đầu
        if k == -1:
            return False
        last = k
    return True

def wrong_end_condition(s, t):
    i = j = 0
    while i < len(s) and j < len(t):
        if s[i] == t[j]:
            i += 1
        j += 1
    return j == len(t)

print("Lỗi 1:", wrong_substring("ace", "abcde"), "(đúng: True)")
print("Lỗi 2:", wrong_presence("aec", "abcde"), wrong_presence("aa", "a"), "(đúng: False False)")
print("Lỗi 3:", wrong_find_no_start("ba", "ab"), "(đúng: False)")
print("Lỗi 4:", wrong_end_condition("axc", "ahbgdc"), "(đúng: False)")

Lỗi 1: False (đúng: True)
Lỗi 2: True True (đúng: False False)
Lỗi 3: True (đúng: False)
Lỗi 4: True (đúng: False)


## 14. Mẹo nhận dạng pattern

- "Giữ **thứ tự tương đối**, được phép **bỏ** phần tử" → subsequence → **hai con trỏ tham lam** (kiểm tra) hoặc **DP** (đếm/tối ưu).
- "Nhiều truy vấn trên cùng một chuỗi dài" → **tiền xử lý** (danh sách vị trí, bảng `next`) + tìm nhị phân.
- Hai con trỏ trên **hai** dãy: mỗi con trỏ chỉ tiến, không lùi → `O(n + m)`.
- Bài anh em: **792 Number of Matching Subsequences** (đúng Follow up), **524 Longest Word in Dictionary through Deleting**, **1143 LCS**, **115 Distinct Subsequences**, **2486 Append Characters to String to Make Subsequence**.

## 15. Biến thể

### 15.1. Cần thêm bao nhiêu ký tự vào cuối `t` để `s` thành dãy con? (LeetCode 2486 — đổi vai)

Chạy hai con trỏ; số ký tự của `s` **chưa khớp** chính là đáp án.

In [17]:
def append_characters(t: str, s: str) -> int:
    i = 0
    for ch in t:
        if i < len(s) and s[i] == ch:
            i += 1
    return len(s) - i

print(append_characters("coaching", "coding"), append_characters("abcde", "a"), append_characters("z", "abcde"))

4 0 5


### 15.2. Đếm bao nhiêu từ trong danh sách là dãy con của `t` (LeetCode 792)

Dùng `SubsequenceChecker` đã tiền xử lý.

In [18]:
def num_matching_subseq(t: str, words: list[str]) -> int:
    checker = SubsequenceChecker(t)
    return sum(checker.check(w) for w in words)      # True được tính là 1

print(num_matching_subseq("abcde", ["a", "bb", "acd", "ace"]))   # 3

3


## 16. Bài tập tự luyện (TODO)

**Bài 1 (tính tay).** Trace hai con trỏ cho `s = "bda"`, `t = "abcdab"`. Kết quả?

**Bài 2 (dự đoán output).** `it = iter("abcabc")`; `["c" in it, "a" in it, "c" in it, "b" in it]` bằng gì?

**Bài 3 (điền chỗ trống).** Hoàn thiện `exercise_two_pointers`.

**Bài 4 (viết lại).** Viết bản **đệ quy**: `rec(i, j)` trả `True` nếu `s[i:]` là dãy con của `t[j:]`.

**Bài 5 (biến thể).** Trả về **danh sách vị trí** trong `t` mà thuật toán tham lam dùng để khớp `s` (hoặc `None` nếu không khớp).

**Bài 6 (mini challenge — LeetCode 524).** Cho `t` và danh sách `words`, trả về từ **dài nhất** là dãy con của `t`; nếu nhiều từ cùng độ dài, chọn từ nhỏ nhất theo thứ tự từ điển; không có thì `""`.

In [19]:
def exercise_two_pointers(s: str, t: str) -> bool:
    # TODO 1: i = j = 0
    # TODO 2: while i < len(s) and j < len(t): khớp thì i += 1; luôn j += 1
    # TODO 3: return i == len(s)
    pass

# assert exercise_two_pointers("abc", "ahbgdc") and not exercise_two_pointers("axc", "ahbgdc")

In [20]:
def exercise_recursive(s: str, t: str) -> bool:
    def rec(i: int, j: int) -> bool:
        # TODO: trường hợp cơ sở i == len(s) -> True; j == len(t) -> False
        # TODO: nếu s[i] == t[j] -> rec(i + 1, j + 1), ngược lại rec(i, j + 1)
        pass
    return rec(0, 0)

# assert exercise_recursive("ace", "abcde") and not exercise_recursive("aec", "abcde")

In [21]:
def greedy_positions(s: str, t: str):
    # TODO
    pass

# assert greedy_positions("abc", "ahbgdc") == [0, 2, 5] and greedy_positions("axc", "ahbgdc") is None

In [22]:
def find_longest_word(t: str, words: list[str]) -> str:
    # TODO
    pass

# assert find_longest_word("abpcplea", ["ale", "apple", "monkey", "plea"]) == "apple"
# assert find_longest_word("abpcplea", ["a", "b", "c"]) == "a"

## 17. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | tìm `b` sớm nhất | rồi `d` sau đó | rồi `a` sau `d` |
| 2 | `in` tiêu thụ iterator | "c" ăn a,b,c | lần sau tiếp từ vị trí 3 |
| 3 | xem mục 4 | điều kiện kép trong `while` | so `i` với `len(s)` |
| 4 | hai trường hợp cơ sở | kiểm `i == len(s)` **trước** | 2 lời gọi đệ quy theo khớp/không |
| 5 | giống hai con trỏ | `append(j)` khi khớp | cuối cùng kiểm `len(res) == len(s)` |
| 6 | dùng lại `is_subseq` | sắp theo `(-len(w), w)` | trả từ đầu tiên là dãy con |

## 18. Đáp án tham khảo + Template + Checklist

**Bài 1:** `b` khớp tại 1, `d` tại 3, `a` tại 4 → `True` (vị trí `[1, 3, 4]`).

**Bài 2:** `[True, True, True, False]` — `"c"` tiêu thụ `a, b, c`; `"a"` tiêu thụ `a` (vị trí 3); `"c"` tiêu thụ `b, c`; tới `"b"` thì iterator đã hết → `False`.

In [23]:
def answer_two_pointers(s, t):
    i = j = 0
    while i < len(s) and j < len(t):
        if s[i] == t[j]:
            i += 1
        j += 1
    return i == len(s)

def answer_recursive(s, t):
    def rec(i, j):
        if i == len(s):
            return True
        if j == len(t):
            return False
        if s[i] == t[j]:
            return rec(i + 1, j + 1)
        return rec(i, j + 1)
    return rec(0, 0)

def answer_greedy_positions(s, t):
    res = []
    j = 0
    for ch in s:
        while j < len(t) and t[j] != ch:
            j += 1
        if j == len(t):
            return None
        res.append(j)
        j += 1
    return res

def answer_find_longest_word(t, words):
    for w in sorted(words, key=lambda w: (-len(w), w)):
        if answer_two_pointers(w, t):
            return w
    return ""

_ = trace_subseq("bda", "abcdab")
print("Bài 1 vị trí:", answer_greedy_positions("bda", "abcdab"))
it = iter("abcabc")
bai2 = ["c" in it, "a" in it, "c" in it, "b" in it]
print("Bài 2:", bai2)
assert bai2 == [True, True, True, False]
assert answer_two_pointers("abc", "ahbgdc") and not answer_two_pointers("axc", "ahbgdc")
assert answer_recursive("ace", "abcde") and not answer_recursive("aec", "abcde")
assert answer_greedy_positions("abc", "ahbgdc") == [0, 2, 5] and answer_greedy_positions("axc", "ahbgdc") is None
assert answer_find_longest_word("abpcplea", ["ale", "apple", "monkey", "plea"]) == "apple"
assert answer_find_longest_word("abpcplea", ["a", "b", "c"]) == "a"
print("Đáp án PASS toàn bộ!")

s='bda', t='abcdab'
  Bước 1: i=0 s[i]='b' | j=0 t[j]='a' -> bỏ qua t[j]
  Bước 2: i=0 s[i]='b' | j=1 t[j]='b' -> KHỚP, i += 1
  Bước 3: i=1 s[i]='d' | j=2 t[j]='c' -> bỏ qua t[j]
  Bước 4: i=1 s[i]='d' | j=3 t[j]='d' -> KHỚP, i += 1
  Bước 5: i=2 s[i]='a' | j=4 t[j]='a' -> KHỚP, i += 1
  Dừng: i=3, j=5 -> đã khớp hết s -> True
Bài 1 vị trí: [1, 3, 4]
Bài 2: [True, True, True, False]
Đáp án PASS toàn bộ!


(Chú ý ở Bài 4: đệ quy sâu tối đa `len(t)` tầng; với `t` dài `10^4` có thể vượt giới hạn đệ quy mặc định của Python (~1000). Vì vậy bản lặp (hai con trỏ) an toàn hơn để nộp.)

### Template nhớ nhanh

```python
class Solution:
    def isSubsequence(self, s: str, t: str) -> bool:
        i = 0                                  # số ký tự của s đã khớp
        for ch in t:                           # j ẩn trong vòng for
            if i < len(s) and s[i] == ch:
                i += 1                         # khớp sớm nhất có thể (tham lam)
        return i == len(s)
```

### Checklist submit
- [ ] Phân biệt subsequence với substring (`in`)?
- [ ] Giữ **thứ tự** và **số lần** (mỗi ký tự của `t` dùng tối đa 1 lần)?
- [ ] `s` rỗng → `True`; `t` rỗng và `s` không rỗng → `False`?
- [ ] Điều kiện kết luận là `i == len(s)`?
- [ ] Trả lời được Follow up: tiền xử lý vị trí + `bisect`, hoặc bảng `next`?